# Video 1: Combined top-down and back camera video trial examples

This notebook generates side-by-side video snippets from top-down and back camera views for selected trials, overlaying trial information and reward status.

*Note: You need to download the back videos and top-down videos for the session to work on.*

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
cd "/app/"

/app


/usr/local/lib/python3.9/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%run env.py
%run run.py connect

2026-01-09 13:28:10,830::INFO::settings.py::Setting loglevel to INFO
2026-01-09 13:28:10,830::INFO::settings.py::Setting stores to {}
2026-01-09 13:28:10,831::INFO::settings.py::Setting database.misc.schema_prefix to 
2026-01-09 13:28:10,831::INFO::settings.py::Setting database.misc.create_tables to True
2026-01-09 13:28:10,832::INFO::settings.py::Setting enable_python_native_blobs to True
2026-01-09 13:28:10,832::INFO::settings.py::Setting database.host to 128.178.51.167:3309
2026-01-09 13:28:10,833::INFO::settings.py::Setting database.user to root
2026-01-09 13:28:10,833::INFO::settings.py::Setting database.password to simple
2026-01-09 13:28:10,875::INFO::connection.py::Connected root@128.178.51.167:3309


Connecting root@128.178.51.167:3309


In [4]:
import cv2
import pickle
import pandas as pd
import numpy as np

In [5]:
# Back camera related files
cam_back_video_path = "/app/back_videos/CamBack_Pheasant_2024-08-15_2.avi"
cam_back_timesteps_path = "/app/back_videos/TIMESTAMPS_CamBack_Pheasant_2024-08-15_2.npy"

# Top-down camera related files
cam_topdown_video_path = "/app/back_videos/Imagingsource_Pheasant_2024-08-15_2_VIDEO.avi"
cam_topdown_times_path = "/app/back_videos/Imagingsource_Pheasant_2024-08-15_2_TS.npy"

In [6]:
with open(cam_back_timesteps_path, 'rb') as f:
    back_timesteps_data = np.load(f)
with open(cam_topdown_times_path, 'rb') as f:
    topdown_times_data = np.load(f)

In [7]:
from vr4mice.schema import vr4mice
start_time_game = (vr4mice.State() & {"dataset": "Pheasant_2024-08-15_2"}).fetch("start_time")[0]

In [8]:
topdown_timesteps = topdown_times_data - start_time_game
back_timesteps = back_timesteps_data - start_time_game

In [9]:
top_df = pd.DataFrame({
    "time_ms": topdown_timesteps,
    "top_frames": range(len(topdown_timesteps))
})
back_df = pd.DataFrame({
    "time_ms": back_timesteps,
    "back_frames": range(len(back_timesteps))
})

In [10]:
aligned_df = pd.merge_asof(
    top_df.sort_values('time_ms'),
    back_df.sort_values('time_ms'),
    on='time_ms',
    direction='nearest'
)

In [11]:
from vr4mice.schema.base_analysis import DataFrame

In [12]:
df = DataFrame().get_data(
    key={"dataset": "Pheasant_2024-08-15_2"},
    columns=[
        "dataset",
        "reward",
        "trial",
        "aperture",
        "iti",
        "trial_left_choice",
        "step",
        "step_time",
        "time_elapsed",
        "x",
        "y",
    ],
)

In [13]:
total_df = pd.merge_asof(df.sort_values('step_time'),
                         aligned_df.sort_values('time_ms'),
                            left_on='step_time',
                            right_on='time_ms',
                            direction='nearest', tolerance=0.5)

from vr4mice.analysis.analysis import get_rewarded
total_df["rewarded_trial"] = get_rewarded(total_df)

In [14]:
import cv2
import numpy as np

def generate_side_by_side_snippets(df, top_path, back_path, output_name, speed_factor=0.5, display_reward=False):
    cap_top = cv2.VideoCapture(top_path)
    cap_back = cv2.VideoCapture(back_path)
    
    master_fps = 100
    output_fps = master_fps * speed_factor 
    
    w1 = int(cap_top.get(cv2.CAP_PROP_FRAME_WIDTH))
    h1 = int(cap_top.get(cv2.CAP_PROP_FRAME_HEIGHT))
    w2 = int(cap_back.get(cv2.CAP_PROP_FRAME_WIDTH))
    h2 = int(cap_back.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Resize logic for vertical alignment
    scale_factor = h1 / h2
    new_w2 = int(w2 * scale_factor)
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_name, fourcc, output_fps, (w1 + new_w2, h1))

    for _, row in df.iterrows():
        # Seek to specific aligned frames
        cap_top.set(cv2.CAP_PROP_POS_FRAMES, int(row['top_frames']))
        cap_back.set(cv2.CAP_PROP_POS_FRAMES, int(row['back_frames']))
        
        ret1, frame1 = cap_top.read()
        ret2, frame2 = cap_back.read()
        
        if ret1 and ret2:
            frame1 = cv2.rotate(frame1, cv2.ROTATE_180)
            frame2_resized = cv2.resize(frame2, (new_w2, h1))
            combined = np.hstack((frame1, frame2_resized))
            
            # Overlay trial info and individual timestamps
            if display_reward:
                reward_info = f"Reward: {int(row['rewarded_trial'])}"
                cv2.putText(combined, reward_info, (20, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0) if row['rewarded_trial'] > 0 else (0, 0, 255), 1)
            info = f"Trial: {int(row['trial'])}"
            cv2.putText(combined, info, (20, h1 - 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
            
            out.write(combined)
            
    cap_top.release()
    cap_back.release()
    out.release()

### Set of both rewarded and failed trials

In [15]:
selected_trials = np.arange(11, 250, 15)
selected_df = total_df[(total_df["iti"]<1) & (total_df["trial"].isin(selected_trials))]

In [16]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, cam_back_video_path, '/app/back_videos/combined_top_back.mp4',
                               display_reward=True)

### Rewarded trials

In [17]:
selected_trials = np.arange(11, 250, 10)
selected_df = total_df[(total_df["iti"]<1) & (total_df["rewarded_trial"]==1) & (total_df["trial"].isin(selected_trials))]

In [18]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, cam_back_video_path, '/app/back_videos/combined_top_back_rewarded.mp4')

### Failed trials

In [19]:
selected_trials = np.arange(11, 250, 10)
selected_df = total_df[(total_df["iti"]<1) & (total_df["rewarded_trial"]==0) & (total_df["trial"].isin(selected_trials))]

In [20]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, cam_back_video_path, '/app/back_videos/combined_top_back_failed.mp4')